# Tutorial: Transmission Chains and Touchstone Models

Audience:
- Users modeling the electrical path between generated waveforms and qubit drives.

Learning goals:
- Load synthetic `.s2p` and `.s5p` networks.
- Propagate IQ and RF traces through single-channel and MIMO transmission chains.
- Inspect stage history at the AWG and qubit reference planes.
- Feed propagated bundle outputs into gate-level Hamiltonian simulations.


## 1. Environment setup

This notebook assumes the local `pysuqu` environment is already available.

The walkthrough focuses on five connected capabilities:

- inserting an explicit `TransmissionChain` between the AWG and the qubit
- loading measured or simulated `.s2p/.s5p` files with `TouchstoneStage`
- selecting custom `Sij` paths such as `S11` and `S31`
- propagating multiple drive lines together with `SignalBundle` and `MIMOTouchstoneStage`
- exporting the qubit-side bundle into QuTiP coefficient functions for a generic multi-drive Hamiltonian


In [ ]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

required = ["numpy", "matplotlib", "qutip"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(
        "Install the runtime dependencies first: " + ", ".join(missing)
    )

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pysuqu").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pysuqu").is_dir():
    raise RuntimeError("Run this notebook from the pysuqu repository checkout.")
sys.path.insert(0, str(PROJECT_ROOT))

from pysuqu.funclib import (
    AttenuatorStage,
    BundleTransmissionChain,
    ChannelSchedule,
    DelayStage,
    EnvelopeParams,
    MIMOTouchstoneStage,
    MixerParams,
    PulseEvent,
    TouchstoneStage,
    TransmissionChain,
    WaveformGenerator,
    load_touchstone_network,
)
from pysuqu.qubit import SingleQubitGate

plt.style.use("seaborn-v0_8-whitegrid")
DATA_DIR = PROJECT_ROOT / "tmp" / "demo_05_touchstone"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR


## 2. Create sample Touchstone files

To keep the notebook self-contained, the next cell writes two small files into `tmp/demo_05_touchstone/`:

- one `s2p` file that behaves like a single-line filter
- one `s5p` file that mimics a multi-port package model


In [ ]:
def write_touchstone_full(path: Path, frequencies: np.ndarray, matrices: np.ndarray, *, option_line: str = '# GHZ S RI R 50', extra_header: list[str] | None = None) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    header = ['! generated by demo_05', option_line]
    if extra_header:
        header.extend(extra_header)

    lines = list(header)
    for freq, matrix in zip(frequencies, matrices):
        values = [f'{float(freq):.12g}']
        for input_port in range(matrix.shape[1]):
            for output_port in range(matrix.shape[0]):
                value = matrix[output_port, input_port]
                values.append(f'{float(np.real(value)):.12g}')
                values.append(f'{float(np.imag(value)):.12g}')
        lines.append(' '.join(values))

    path.write_text('\n'.join(lines), encoding='utf-8')
    return path

s2p_freq = np.array([4.6, 4.8, 5.0, 5.2, 5.4])
s2p_matrix = np.zeros((len(s2p_freq), 2, 2), dtype=np.complex128)
s2p_matrix[:, 0, 0] = np.array([0.10, 0.09, 0.08, 0.08, 0.07])  # S11
s2p_matrix[:, 1, 0] = np.array([0.95, 0.85, 0.70, 0.50, 0.35])  # S21
s2p_matrix[:, 0, 1] = np.array([0.03, 0.03, 0.02, 0.02, 0.02])  # S12
s2p_matrix[:, 1, 1] = np.array([0.12, 0.10, 0.09, 0.09, 0.08])  # S22
s2p_path = write_touchstone_full(DATA_DIR / 'demo_filter.s2p', s2p_freq, s2p_matrix)

s5p_freq = np.array([4.7, 4.9, 5.1, 5.3])
s5p_matrix = np.zeros((len(s5p_freq), 5, 5), dtype=np.complex128)
s5p_matrix[:, 2, 0] = np.array([0.30, 0.24, 0.18, 0.12])  # S31
s5p_matrix[:, 0, 0] = np.array([0.08, 0.07, 0.06, 0.05])  # S11
s5p_matrix[:, 1, 0] = np.array([0.65, 0.60, 0.52, 0.40])  # S21
s5p_path = write_touchstone_full(DATA_DIR / 'demo_package.s5p', s5p_freq, s5p_matrix)

mimo_freq = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
mimo_matrix = np.zeros((len(mimo_freq), 2, 2), dtype=np.complex128)
mimo_matrix[:, 0, 0] = np.array([1.00, 0.98, 0.95, 0.92, 0.90])       # direct A -> A
mimo_matrix[:, 1, 1] = np.array([0.90, 0.88, 0.86, 0.83, 0.80])       # direct B -> B
mimo_matrix[:, 1, 0] = np.array([0.20, 0.18, 0.16, 0.14, 0.12])       # leakage A -> B
mimo_matrix[:, 0, 1] = np.array([0.35, 0.32, 0.28, 0.24, 0.20]) * np.exp(1j * 0.25)  # leakage B -> A
mimo_path = write_touchstone_full(DATA_DIR / 'demo_mimo_crosstalk.s2p', mimo_freq, mimo_matrix)

print('Created:', s2p_path)
print('Created:', s5p_path)
print('Created:', mimo_path)


## 3. Load S-parameter networks directly

`load_touchstone_network(...)` parses a Touchstone file into a frequency axis plus a dense complex S-parameter tensor.

Before propagating waveforms, we first inspect the imported `S21`, `S31`, and `S11` data directly.


In [ ]:
network_s2 = load_touchstone_network(s2p_path)
network_s5 = load_touchstone_network(s5p_path)

print('s2p ports:', network_s2.n_ports)
print('s2p frequencies (GHz):', network_s2.frequencies)
print('S21 magnitude:', np.round(np.abs(network_s2.get_response(output_port=2, input_port=1)), 4))
print('S11 magnitude:', np.round(np.abs(network_s2.get_response(output_port=1, input_port=1)), 4))
print()
print('s5p ports:', network_s5.n_ports)
print('S31 magnitude:', np.round(np.abs(network_s5.get_response(output_port=3, input_port=1)), 4))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(network_s2.frequencies, np.abs(network_s2.get_response(2, 1)), marker='o', label='|S21|')
ax[0].plot(network_s2.frequencies, np.abs(network_s2.get_response(1, 1)), marker='s', label='|S11|')
ax[0].set_title('Demo s2p')
ax[0].set_xlabel('Frequency (GHz)')
ax[0].set_ylabel('Magnitude')
ax[0].legend()

ax[1].plot(network_s5.frequencies, np.abs(network_s5.get_response(3, 1)), marker='o', label='|S31|')
ax[1].plot(network_s5.frequencies, np.abs(network_s5.get_response(2, 1)), marker='s', label='|S21|')
ax[1].set_title('Demo s5p')
ax[1].set_xlabel('Frequency (GHz)')
ax[1].legend()

fig.tight_layout()


## 4. Insert `TouchstoneStage` into a `TransmissionChain`

The next cells build a simplified single-qubit drive channel:

- an IQ waveform leaves the AWG
- a fixed attenuator is applied
- a `TouchstoneStage` models the measured filter response
- an explicit delay models cable latency

This chain can be attached directly to `ChannelSchedule.transmission_chain`, or passed as a temporary override into `run_simulation(...)`.


In [ ]:
generator = WaveformGenerator(total_time=32.0, sample_rate=2.0)

pulse = PulseEvent(
    start_time=6.0,
    name='x90_like',
    if_freq=0.05,
    envelope=EnvelopeParams(
        name='cosine_env',
        duration=12.0,
        peak_amp=0.03,
        shape_type='cosine',
    ),
)

channel = ChannelSchedule(
    name='XY_Q1',
    sampling_rate=2.0,
    mixer_config=MixerParams(lo_freq=5.0),
    events=[pulse],
)

touchstone_chain = TransmissionChain(
    name='xy_touchstone_line',
    stages=[
        AttenuatorStage(loss_db=3.0, name='room_temp_loss'),
        TouchstoneStage(file_path=s2p_path, input_port=1, output_port=2, name='package_filter'),
        DelayStage(delay_ns=1.5, name='cable_delay'),
    ],
)

channel_with_line = channel.clone_with(transmission_chain=touchstone_chain)
print(touchstone_chain.describe())
channel_with_line.display()


## 5. Compare AWG-side and qubit-side waveforms

`generate_awg_output(...)` returns the waveform on the AWG reference plane.
`generate_qubit_output(...)` returns the waveform seen by the qubit after the chain.

We keep the intermediate stage history to see how every block changes the IQ signal.


For an interactive Plotly view, call `generator.plot_schedule(channel_with_line, plane="qubit", capture_history=True)`.

In [ ]:
awg_trace = generator.generate_awg_output(channel, mode='iq')
qubit_result = generator.generate_qubit_output(channel_with_line, mode='iq', capture_history=True)

print('AWG plane:', awg_trace.plane, 'domain:', awg_trace.domain)
print('Qubit plane:', qubit_result.output_trace.plane, 'domain:', qubit_result.output_trace.domain)
print('Stage count:', len(qubit_result.stage_outputs))
[(trace.metadata.get('last_stage'), trace.plane) for trace in qubit_result.stage_outputs]


In [ ]:
stage_traces = [qubit_result.input_trace, *qubit_result.stage_outputs]
stage_labels = ['AWG input'] + [trace.metadata.get('last_stage', trace.label) for trace in qubit_result.stage_outputs]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for trace, label in zip(stage_traces, stage_labels):
    axes[0].plot(trace.t_axis, np.real(trace.values), label=label)
    axes[1].plot(trace.t_axis, np.imag(trace.values), label=label)

axes[0].set_title('Transmission chain history on the IQ plane')
axes[0].set_ylabel('I amplitude')
axes[1].set_ylabel('Q amplitude')
axes[1].set_xlabel('Time (ns)')
axes[0].legend(loc='upper right', ncol=2)
fig.tight_layout()

peak_awg = float(np.max(np.abs(awg_trace.values)))
peak_qubit = float(np.max(np.abs(qubit_result.output_trace.values)))
print(f'Peak |IQ| ratio (qubit / awg): {peak_qubit / peak_awg:.3f}')


## 6. Select arbitrary `Sij` paths from a multi-port file

For an `s5p` package model, the useful path is not always the default `S21`.

The next cell applies the same AWG IQ waveform to two different paths from the same file:

- `S31`, which can represent a package cross-talk path
- `S11`, which can represent a reflection path

The only change is `input_port` and `output_port`.


In [ ]:
s31_stage = TouchstoneStage(file_path=s5p_path, input_port=1, output_port=3, name='package_s31')
s11_stage = TouchstoneStage(file_path=s5p_path, input_port=1, output_port=1, name='package_s11')

s31_trace = s31_stage.apply(awg_trace)
s11_trace = s11_stage.apply(awg_trace)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(awg_trace.t_axis, np.abs(awg_trace.values), label='|AWG IQ|', linewidth=2)
ax.plot(s31_trace.t_axis, np.abs(s31_trace.values), label='|After S31|')
ax.plot(s11_trace.t_axis, np.abs(s11_trace.values), label='|After S11|')
ax.set_title('Custom port-path selection from an s5p file')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('|IQ|')
ax.legend()
fig.tight_layout()

print('S31 peak ratio:', float(np.max(np.abs(s31_trace.values)) / np.max(np.abs(awg_trace.values))))
print('S11 peak ratio:', float(np.max(np.abs(s11_trace.values)) / np.max(np.abs(awg_trace.values))))


## 7. Multi-line MIMO propagation with a bundle chain

A real package or filter can mix multiple microwave lines. `SignalBundle` keeps those aligned waveforms together, and `MIMOTouchstoneStage` applies the selected S-parameter submatrix in one step.


In [ ]:
mimo_generator = WaveformGenerator(total_time=12.0, sample_rate=1.0)

pulse_a = PulseEvent(
    start_time=1.0,
    name='drive_a_square',
    if_freq=0.0,
    envelope=EnvelopeParams(name='drive_a_env', duration=8.0, peak_amp=1.0, shape_type='square'),
)
pulse_b = PulseEvent(
    start_time=3.0,
    name='drive_b_square',
    if_freq=0.0,
    envelope=EnvelopeParams(name='drive_b_env', duration=6.0, peak_amp=0.6, shape_type='square'),
)

drive_a = ChannelSchedule(
    name='drive_a',
    sampling_rate=1.0,
    mixer_config=MixerParams(lo_freq=0.0),
    mixer_correction=False,
    events=[pulse_a],
)
drive_b = ChannelSchedule(
    name='drive_b',
    sampling_rate=1.0,
    mixer_config=MixerParams(lo_freq=0.0),
    mixer_correction=False,
    events=[pulse_b],
)

mimo_chain = BundleTransmissionChain(
    name='two_line_package',
    stages=[
        MIMOTouchstoneStage(
            file_path=mimo_path,
            input_ports=(1, 2),
            output_ports=(1, 2),
            input_channels=('drive_a', 'drive_b'),
            output_channels=('qubit_a', 'qubit_b'),
            frequency_mode='relative',
            name='measured_package_mimo',
        )
    ],
)

mimo_schedules = {'drive_a': drive_a, 'drive_b': drive_b}
awg_bundle = mimo_generator.generate_awg_bundle(mimo_schedules, mode='rf')
qubit_bundle = mimo_generator.generate_qubit_bundle(mimo_schedules, mode='rf', chain=mimo_chain)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for name in awg_bundle.order:
    axes[0].plot(awg_bundle[name].t_axis, awg_bundle[name].values, label=name)
for name in qubit_bundle.order:
    axes[1].plot(qubit_bundle[name].t_axis, qubit_bundle[name].values, label=name)

axes[0].set_title('AWG-side independent drive lines')
axes[1].set_title('Qubit-side lines after MIMO S-parameter propagation')
axes[1].set_xlabel('Time (ns)')
for ax in axes:
    ax.set_ylabel('RF amplitude')
    ax.legend(loc='upper right')
fig.tight_layout()

print('AWG bundle channels:', awg_bundle.order)
print('Qubit bundle channels:', qubit_bundle.order)
print('Peak qubit_a:', float(np.max(np.abs(qubit_bundle['qubit_a'].values))))
print('Peak qubit_b:', float(np.max(np.abs(qubit_bundle['qubit_b'].values))))


## 8. Feed the qubit-side bundle into a multi-drive Hamiltonian

The new gate-level multi-drive facade keeps the workflow compact: build the propagated bundle, pair each output line with its operator, and call `run_multidrive_simulation(...)`.


In [ ]:
multidrive_gate = SingleQubitGate(
    total_time=12.0,
    sample_rate=1.0,
    qubit_frequency=5.0,
    qubit_anharmonicity=-0.25,
    energy_trunc_level=3,
)

identity = qt.qeye(2)
static_hamiltonian = 0.0 * qt.tensor(identity, identity)
drive_operators = {
    'qubit_a': qt.tensor(qt.sigmax(), identity),
    'qubit_b': qt.tensor(identity, qt.sigmax()),
}

h_total, drive_funcs = multidrive_gate.build_multidrive_hamiltonian(
    schedules=mimo_schedules,
    drive_operators=drive_operators,
    transmission_chain=mimo_chain,
    mode='rf',
    plane='qubit',
    static_hamiltonian=static_hamiltonian,
)

psi0 = qt.tensor(qt.basis(2, 0), qt.basis(2, 0))
result_mimo = multidrive_gate.run_multidrive_simulation(
    schedules=mimo_schedules,
    drive_operators=drive_operators,
    initial_state=psi0,
    transmission_chain=mimo_chain,
    mode='rf',
    plane='qubit',
    static_hamiltonian=static_hamiltonian,
)

projector_q0_excited = qt.tensor(qt.basis(2, 1) * qt.basis(2, 1).dag(), identity)
projector_q1_excited = qt.tensor(identity, qt.basis(2, 1) * qt.basis(2, 1).dag())
pop_q0 = np.asarray([np.real(qt.expect(projector_q0_excited, state)) for state in result_mimo.states], dtype=float)
pop_q1 = np.asarray([np.real(qt.expect(projector_q1_excited, state)) for state in result_mimo.states], dtype=float)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(result_mimo.times, pop_q0, label='P(q0=1), driven by qubit_a')
ax.plot(result_mimo.times, pop_q1, label='P(q1=1), driven by qubit_b')
ax.set_title('Toy two-qubit evolution driven by propagated MIMO bundle outputs')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Excited-state population')
ax.legend()
fig.tight_layout()

print('Hamiltonian term count:', len(h_total))
print('Drive function keys:', tuple(drive_funcs))
print(f'Final P(q0=1): {pop_q0[-1]:.6f}')
print(f'Final P(q1=1): {pop_q1[-1]:.6f}')


## 9. Run gate dynamics with and without the transmission chain


In [ ]:
gate = SingleQubitGate(
    total_time=32.0,
    sample_rate=2.0,
    qubit_frequency=5.0,
    qubit_anharmonicity=-0.25,
    energy_trunc_level=3,
    pulse_channel=channel,
)

result_ideal = gate.run_simulation(channel=channel)
result_line = gate.run_simulation(channel=channel, transmission_chain=touchstone_chain)

projector_1 = qt.basis(3, 1) * qt.basis(3, 1).dag()
pop_ideal = np.asarray([np.real(qt.expect(projector_1, state)) for state in result_ideal.states], dtype=float)
pop_line = np.asarray([np.real(qt.expect(projector_1, state)) for state in result_line.states], dtype=float)

print(f'Final P(|1>) without line: {pop_ideal[-1]:.6f}')
print(f'Final P(|1>) with line:    {pop_line[-1]:.6f}')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(result_ideal.times, pop_ideal, label='Ideal line')
ax.plot(result_line.times, pop_line, label='Touchstone line')
ax.set_title('SingleQubitGate population dynamics with and without the transmission chain')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('P(|1>)')
ax.legend()
fig.tight_layout()


## 10. Practical notes and extension ideas

Useful conventions in the current implementation:

- `TouchstoneStage` defaults to `S21`, which means `input_port=1, output_port=2`
- choose `input_port` and `output_port` explicitly for paths such as `S11`, `S31`, or other measured package couplings
- `MIMOTouchstoneStage` uses `input_ports` and `output_ports` to select a full S-parameter submatrix, then maps input channel names to output channel names
- `WaveformGenerator.get_qutip_bundle_funcs(...)` exports qubit-side bundle outputs as `{channel_name: coeff_func}` for multi-drive Hamiltonians
- `iq_complex` defaults to absolute-frequency evaluation, so `lo_freq + f_if` matters
- for `iq_complex` MIMO bundles, all traces in the bundle should share one LO frequency
- the current parser expects **full matrix** `.sNp` data; lower/upper matrix formats will raise a clear error
- if the measured frequency span is narrower than the simulated waveform spectrum, choose `out_of_band='edge' | 'zero' | 'error'` explicitly

### Small exercise

Change `output_port` in the next cell to `1`, `2`, or `3`, and compare how the same `s5p` file changes the waveform.


In [ ]:
exercise_stage = TouchstoneStage(
    file_path=s5p_path,
    input_port=1,
    output_port=3,
    name='exercise_stage',
)

exercise_trace = exercise_stage.apply(awg_trace)
print('Selected path:', exercise_stage.describe())
print('Peak ratio:', float(np.max(np.abs(exercise_trace.values)) / np.max(np.abs(awg_trace.values))))
